<a href="https://colab.research.google.com/github/peperjet/bc-ml/blob/main/movie_admissions/movie_admissions_260402.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 라이브러리 및 데이터 불러오기


In [49]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

# 데이터 불러오기 (파일 이름 확인 필수)
train = pd.read_csv('/content/movies_train.csv')
test = pd.read_csv('/content/movies_test.csv')
submission = pd.read_csv('/content/submission.csv')

2. 자료분석 (EDA)

In [50]:
# 상위 데이터 확인
train.head()

,title,distributor,genre,release_time,time,screening_rat,director,dir_prev_bfnum,dir_prev_num,num_staff,num_actor,box_off_num
0,개들의 전쟁,롯데엔터테인먼트,액션,2012-11-22,96,청소년 관람불가,조병옥,NaN,0,91,2,23398
1,내부자들,(주)쇼박스,느와르,2015-11-19,130,청소년 관람불가,우민호,1161602.5,2,387,3,7072501
2,은밀하게 위대하게,(주)쇼박스,액션,2013-06-05,123,15세 관람가,장철수,220775.2,4,343,4,6959083
3,나는 공무원이다,(주)NEW,코미디,2012-07-12,101,전체 관람가,구자홍,23894.0,2,20,6,217866
4,불량남녀,쇼박스(주)미디어플렉스,코미디,2010-11-04,108,15세 관람가,신근호,1.0,1,251,2,483387


*   title : 영화의 제목
*   distributor : 배급사
*   genre : 장르
*   release_time : 개봉일
*   time : 상영시간(분)
*   screening_rat : 상영등급
*   director : 감독이름
*   dir_prev_bfnum : 해당 감독이 이 영화를 만들기 전 제작에 참여한 영화에서의 평균 관객수(단 관객수가 알려지지 않은 영화 제외)
*   dir_prev_num : 해당 감독이 이 영화를 만들기 전 제작에 참여한 *   영화의 개수(단 관객수가 알려지지 않은 영화 제외)
*   num_staff : 스텝수
*   num_actor : 주연배우수
*   box_off_num : 관객수

In [51]:
# 요약 정보 및 결측치 확인
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           600 non-null    object 
 1   distributor     600 non-null    object 
 2   genre           600 non-null    object 
 3   release_time    600 non-null    object 
 4   time            600 non-null    int64  
 5   screening_rat   600 non-null    object 
 6   director        600 non-null    object 
 7   dir_prev_bfnum  270 non-null    float64
 8   dir_prev_num    600 non-null    int64  
 9   num_staff       600 non-null    int64  
 10  num_actor       600 non-null    int64  
 11  box_off_num     600 non-null    int64  
dtypes: float64(1), int64(5), object(6)
memory usage: 56.4+ KB


In [52]:
# 기술 통계량 확인 (실수 소스점 1자리까지)
pd.options.display.float_format = '{:.1f}'.format
train.describe()

,time,dir_prev_bfnum,dir_prev_num,num_staff,num_actor,box_off_num
count,600.0,270.0,600.0,600.0,600.0,600.0
mean,100.9,1050442.9,0.9,151.1,3.7,708181.8
std,18.1,1791408.3,1.2,165.7,2.4,1828005.9
min,45.0,1.0,0.0,0.0,0.0,1.0
25%,89.0,20380.0,0.0,17.0,2.0,1297.2
50%,100.0,478423.6,0.0,82.5,3.0,12591.0
75%,114.0,1286568.6,2.0,264.0,4.0,479886.8
max,180.0,17615314.0,5.0,869.0,25.0,14262766.0


In [53]:
# 장르별 관객 수 평균 확인 (상관관계 파악 예시)
train.groupby('genre')['box_off_num'].mean().sort_values(ascending=False)

# 상관관계 확인
train.corr(numeric_only=True)

,time,dir_prev_bfnum,dir_prev_num,num_staff,num_actor,box_off_num
time,1.0,0.3,0.3,0.6,0.1,0.4
dir_prev_bfnum,0.3,1.0,0.1,0.3,0.1,0.3
dir_prev_num,0.3,0.1,1.0,0.5,0.0,0.3
num_staff,0.6,0.3,0.5,1.0,0.1,0.5
num_actor,0.1,0.1,0.0,0.1,1.0,0.1
box_off_num,0.4,0.3,0.3,0.5,0.1,1.0


3. 데이터 전처리
* 값(결측치)을 처리하고 문자-> 숫자 변경. (dir_prev_bfnum의 결측치는 감독 이전 기록이 없는 것이므로 0으로 채우는 경우가 많다)

In [54]:
# 결측치 확인
train.isna().sum()

,0
title,0
distributor,0
genre,0
release_time,0
time,0
screening_rat,0
director,0
dir_prev_bfnum,330
dir_prev_num,0
num_staff,0


In [55]:
# 결측치 채우기 (이전 관객수 기록이 없으면 0으로 간주)
train['dir_prev_bfnum'] = train['dir_prev_bfnum'].fillna(0)
test['dir_prev_bfnum'] = test['dir_prev_bfnum'].fillna(0)


# 범주형 데이터(장르, 상영등급) 변환
# 모델 학습을 위해 문자열을 숫자로 라벨 인코딩합니다.
train['genre'] = train['genre'].map({'액션':0, '느와르':1, '코미디':2, '다큐멘터리':3, '뮤지컬':4, '드라마':5, '멜로/로맨스':6, '공포':7, '서스펜스':8, '애니메이션':9, '서부극':10, 'SF':11})
test['genre'] = test['genre'].map({'액션':0, '느와르':1, '코미디':2, '다큐멘터리':3, '뮤지컬':4, '드라마':5, '멜로/로맨스':6, '공포':7, '서스펜스':8, '애니메이션':9, '서부극':10, 'SF':11})

4. 변수선택 및 모델 구축

In [56]:
# 예측에 사용할 사진들
features = ['time', 'dir_prev_bfnum', 'dir_prev_num', 'num_staff', 'num_actor', 'genre']
target = 'box_off_num'

X_train = train[features]
y_train = train[target]
X_test = test[features]

In [57]:
print(X_train.shape, y_train.shape)
print(X_test.shape)

(600, 6) (600,)
(243, 6)


In [58]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   time            600 non-null    int64  
 1   dir_prev_bfnum  600 non-null    float64
 2   dir_prev_num    600 non-null    int64  
 3   num_staff       600 non-null    int64  
 4   num_actor       600 non-null    int64  
 5   genre           583 non-null    float64
dtypes: float64(2), int64(4)
memory usage: 28.3 KB


In [59]:
# genre의 결측치를 최빈값으로 채우기
train['genre'] = train['genre'].fillna(train['genre'].mode()[0])
test['genre'] = test['genre'].fillna(test['genre'].mode()[0])

# 2. X_train, X_test 다시 업데이트 (결측치가 채워진 데이터를 반영)
X_train = train[features]
X_test = test[features]

# 3. 확인 (이제 모두 600 non-null이 나와야 합니다)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   time            600 non-null    int64  
 1   dir_prev_bfnum  600 non-null    float64
 2   dir_prev_num    600 non-null    int64  
 3   num_staff       600 non-null    int64  
 4   num_actor       600 non-null    int64  
 5   genre           600 non-null    float64
dtypes: float64(2), int64(4)
memory usage: 28.3 KB


5. 모델학습 및 검증 (K-Fold & LightGBM)

In [60]:
# 교차검증준비
kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [61]:
# 각 폴드별 모델과 예측값을 저장할 리스트
models = []

In [62]:
# 학습 시작
# kf.split(X_train)은 데이터를 5세트의 (학습용 index, 검증용 index)로 나눠줍니다.
for train_index, val_index in kf.split(X_train):
    X_t, X_v = X_train.iloc[train_index], X_train.iloc[val_index]
    y_t, y_v = y_train.iloc[train_index], y_train.iloc[val_index]

    # LightGBM 회귀 모델 설정
    model = lgb.LGBMRegressor(n_estimators=1000, random_state=42)

    # 모델 학습 (검증 데이터를 활용해 조기 종료 설정)
    model.fit(X_t, y_t,
              eval_set=[(X_v, y_v)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(stopping_rounds=100)])

    models.append(model)
    print("--- 한 개의 폴드 학습 완료 ---")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000507 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 317
[LightGBM] [Info] Number of data points in the train set: 480, number of used features: 6
[LightGBM] [Info] Start training from score 723519.229167
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light

In [ ]:
import pandas as pd

# 예측값 상위 10개 출력
print("Predicted Box Office Numbers (Top 10):\n", pd.Series(final_preds).nlargest(10))

# 예측값 하위 10개 출력
print("\nPredicted Box Office Numbers (Bottom 10):\n", pd.Series(final_preds).nsmallest(10))

# 실제 box_off_num의 기술 통계량과 분포 확인
print("\nActual Box Office Numbers (Training Data Description):\n")
display(train['box_off_num'].describe())

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(train['box_off_num'], bins=50, kde=True)
plt.title('Distribution of Actual Box Office Numbers in Training Data')
plt.xlabel('Box Office Number')
plt.ylabel('Frequency')
plt.show()

6. 결과 예측, 제출파일 생성

In [64]:
# 테스트 데이터 예측
preds = []
for model in models:
    preds.append(model.predict(X_test))

# 5개 모델의 예측값 평균 계산
final_preds = np.mean(preds, axis=0)

# 제출 파일(submission.csv)에 채우기
submission['box_off_num'] = final_preds

# 최종 파일 저장 (코랩 왼쪽 파일 탐색기에서 다운로드 가능)
submission.to_csv('movie_submission_v1.csv', index=False)


예측 및 파일 저장이 완료되었습니다!


In [65]:
from google.colab import files

files.download('movie_submission_v1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>